# Proyecto 1 — Pipeline base: Anthropic Economic Index + O*NET

Carga las dos fuentes de datos, las empareja por código de ocupación, y arma las
dos tablas que necesita Phase 1: la de **entrenamiento** (ocupaciones con
`observed_exposure` real) y la de **predicción** (ocupaciones sin puntaje, que es
el objetivo real del proyecto).

Corre esto en **Google Colab** — necesita acceso abierto a internet para bajar los
archivos de Hugging Face y GitHub.

Ver `projects/project-1-exposure-score/phase0.md` y `phase1-predictors.md` en este
mismo repo para el contexto completo de por qué se decidieron estas variables.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)


## 1. Cargar las fuentes

Dos grupos de archivos:
- **La respuesta** (Anthropic Economic Index, hosteado en Hugging Face): `observed_exposure`
  para ~900 ocupaciones, y el detalle de penetración por tarea.
- **Las pistas** (release de O*NET / Anthropic en GitHub): texto de tareas, salario,
  empleo, y la estructura de códigos SOC — disponibles para todo el catálogo, no
  solo las 900 ocupaciones con puntaje.


In [ ]:
ANTHROPIC_HF_BASE = "https://huggingface.co/datasets/Anthropic/EconomicIndex/resolve/main/labor_market_impacts"
GITHUB_MIRROR_BASE = "https://raw.githubusercontent.com/gianluigilopardo/AnthropicEconomicIndex/main"

urls = {
    "job_exposure": f"{ANTHROPIC_HF_BASE}/job_exposure.csv",
    "task_penetration": f"{ANTHROPIC_HF_BASE}/task_penetration.csv",
    "onet_task_mappings": f"{GITHUB_MIRROR_BASE}/onet_task_mappings.csv",
    "onet_task_statements": f"{GITHUB_MIRROR_BASE}/onet_task_statements.csv",
    "automation_vs_augmentation": f"{GITHUB_MIRROR_BASE}/automation_vs_augmentation.csv",
    "soc_structure": f"{GITHUB_MIRROR_BASE}/SOC_Structure.csv",
    "wage_data": f"{GITHUB_MIRROR_BASE}/wage_data.csv",
    "bls_employment": f"{GITHUB_MIRROR_BASE}/bls_employment_may_2023.csv",
}


In [ ]:
job_exposure = pd.read_csv(urls["job_exposure"])
task_penetration = pd.read_csv(urls["task_penetration"])

print("job_exposure:", job_exposure.shape)
display(job_exposure.head())
print("task_penetration:", task_penetration.shape)
display(task_penetration.head())


In [ ]:
onet_task_statements = pd.read_csv(urls["onet_task_statements"])
wage_data = pd.read_csv(urls["wage_data"])
soc_structure = pd.read_csv(urls["soc_structure"])
bls_employment = pd.read_csv(urls["bls_employment"])
onet_task_mappings = pd.read_csv(urls["onet_task_mappings"])

for name, df in [
    ("onet_task_statements", onet_task_statements),
    ("wage_data", wage_data),
    ("soc_structure", soc_structure),
    ("bls_employment", bls_employment),
    ("onet_task_mappings", onet_task_mappings),
]:
    print(f"{name}: {df.shape}")


## 2. El problema real al emparejar códigos

`job_exposure.csv` usa códigos SOC agregados (ej. `13-2011.00` = "Accountants and
Auditors"). Pero `onet_task_statements.csv` casi nunca tiene ese código agregado —
tiene las variantes detalladas de O*NET-SOC (`13-2011.01`, `13-2011.02`, ...).

Esto se verificó de verdad contra los archivos reales: `13-2011.00` **no existe**
en `onet_task_statements`, solo `13-2011.01` y `13-2011.02`. Emparejar por el
código completo pierde silenciosamente casi todas las ocupaciones.

La solución: usar el **prefijo SOC de 7 caracteres** (`13-2011`, sin el sufijo) como
llave de unión, y agregar (concatenar texto, promediar salario) todas las variantes
detalladas que caen bajo ese mismo prefijo.


In [ ]:
def soc_prefix(series):
    return series.astype(str).str.slice(0, 7)

job_exposure["soc_prefix"] = soc_prefix(job_exposure["occ_code"])
onet_task_statements["soc_prefix"] = soc_prefix(onet_task_statements["O*NET-SOC Code"])
wage_data["soc_prefix"] = soc_prefix(wage_data["SOCcode"])

# cuántas ocupaciones de job_exposure encuentran match por código completo vs por prefijo
exact_match = job_exposure["occ_code"].isin(onet_task_statements["O*NET-SOC Code"])
prefix_match = job_exposure["soc_prefix"].isin(onet_task_statements["soc_prefix"])
print(f"Match por codigo completo: {exact_match.sum()} / {len(job_exposure)}")
print(f"Match por prefijo SOC:     {prefix_match.sum()} / {len(job_exposure)}")


## 3. Agregar las features por ocupación (nivel de prefijo SOC)

- **Texto de tareas** → se concatena todo el texto de `Task` por prefijo. Esto es lo
  que luego se convierte en embeddings (variable **derivada**).
- **Salario y JobZone** → se promedian por prefijo (variables **explícitas**, ya
  numéricas, de `wage_data.csv`).

`task_penetration` se carga para referencia (Capa 1, sin modelar) pero
**deliberadamente no se usa como predictor** — ver `phase1-predictors.md` para el
razonamiento sobre por qué sería casi circular.


In [ ]:
task_text = (
    onet_task_statements
    .groupby("soc_prefix")["Task"]
    .apply(lambda s: " ".join(s.astype(str)))
    .reset_index()
    .rename(columns={"Task": "task_text_blob"})
)

wage_agg = (
    wage_data
    .groupby("soc_prefix")
    .agg(median_salary=("MedianSalary", "mean"), job_zone=("JobZone", "mean"))
    .reset_index()
)

features = task_text.merge(wage_agg, on="soc_prefix", how="outer")
print("features (todas las ocupaciones con al menos una fuente):", features.shape)
features.head()


## 4. Armar las dos tablas de Phase 1

- **train**: ocupaciones de `job_exposure` que sí tienen `observed_exposure`, con
  sus features unidas — esto es con lo que se entrena el modelo.
- **predict**: todas las ocupaciones del universo de O*NET/wage que **no** están en
  `job_exposure` — este es el objetivo real del proyecto, predecirles el puntaje.


In [ ]:
train_table = job_exposure.merge(features, on="soc_prefix", how="left")
print("train_table:", train_table.shape)
print("filas sin ninguna feature (revisar):", train_table["task_text_blob"].isna().sum())
train_table.head()


In [ ]:
predict_table = features[~features["soc_prefix"].isin(job_exposure["soc_prefix"])].copy()
print("predict_table (ocupaciones sin puntaje, el objetivo real):", predict_table.shape)
predict_table.head()


## 5. Guardar para el notebook de modelado

Deja `train_table.csv` y `predict_table.csv` en `data/processed/` dentro del repo,
listos para el notebook que entrena el modelo y calcula el MAE contra el baseline.


In [ ]:
import os
os.makedirs("data/processed", exist_ok=True)
train_table.to_csv("data/processed/project1_train.csv", index=False)
predict_table.to_csv("data/processed/project1_predict.csv", index=False)
print("Guardado.")


## Próximos pasos (no incluidos en este notebook)

- Convertir `task_text_blob` en embeddings (o TF-IDF) — ahora es texto crudo.
- Normalizar `median_salary` / `job_zone` (estandarizar o 0–1) si el modelo que se
  use no es basado en árboles.
- Entrenar con validación cruzada sobre `train_table`, reportar MAE contra el
  baseline (predecir el promedio), y aplicar el modelo a `predict_table`.
